# Tiền Xử Lý Chuỗi EAR Cho Mô Hình Hướng 2 (Bảo's Pipeline)

Notebook này thực hiện toàn bộ các bước tiền xử lý cho chuỗi dữ liệu EAR (Eye Aspect Ratio):
1. **Chuẩn hóa Min-Max EAR theo từng cá nhân (Per Person)**
2. **Nội suy phục hồi (Linear Interpolation)** (Điền vào các frame bị mất)
3. **Cắt cửa sổ trượt (Sliding Window N=7)**
4. **Gán nhãn V-Shape Labeling (Blink / No-Blink / Long-Closure)**

Dựa trên tham số chốt từ file EDA của Hiền:
- `Window Size` = 7 frames.
- `EAR Threshold` = 0.1050.

In [ ]:
import os
import numpy as np
import pandas as pd

def process_sequences(input_csv, output_dir, window_size=7, ear_threshold=0.1050):
    print(f"1. Đang tải dữ liệu từ {input_csv}...")
    df = pd.read_csv(input_csv)
    
    if 'status' in df.columns:
        df = df[df['status'] == 'success'].copy()
        
    df = df.drop_duplicates(subset=['video_id', 'frame_index']).copy()
    df['member_id'] = df['video_id'].apply(lambda x: str(x).split('_')[0])
    
    print("2. Đang chuẩn hóa Min-Max EAR theo từng thành viên...")
    df['ear_norm'] = df.groupby('member_id')['ear_avg'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
    
    print("3. Đang nội suy phục hồi các frame bị khuyết...")
    df_interp_list = []
    for video_id, group in df.groupby('video_id'):
        group = group.sort_values('frame_index').set_index('frame_index')
        full_idx = range(group.index.min(), group.index.max() + 1)
        group = group.reindex(full_idx)
        group['ear_avg'] = group['ear_avg'].interpolate(method='linear')
        group['ear_norm'] = group['ear_norm'].interpolate(method='linear')
        group['video_id'] = video_id
        df_interp_list.append(group.reset_index())
        
    df_interp = pd.concat(df_interp_list, ignore_index=True)
    df_interp = df_interp.dropna(subset=['ear_avg'])
    
    print(f"4. Đang trích xuất cửa sổ trượt (N={window_size}) và dán nhãn...")
    X = []
    y = []
    
    for video_id, group in df_interp.groupby('video_id'):
        group = group.sort_values('frame_index')
        ear_values = group['ear_avg'].values
        ear_norm_values = group['ear_norm'].values
        
        if len(ear_values) < window_size:
            continue
            
        for i in range(len(ear_values) - window_size + 1):
            window_ear = ear_values[i:i+window_size]
            window_feat = ear_norm_values[i:i+window_size]
            
            closed_frames = window_ear < ear_threshold
            closed_count = np.sum(closed_frames)
            
            if closed_count > 4:
                label = 2
            elif 2 <= closed_count <= 4 and not closed_frames[0] and not closed_frames[-1]:
                label = 1
            else:
                label = 0
                
            X.append(window_feat)
            y.append(label)
            
    X = np.array(X)[..., np.newaxis] 
    y = np.array(y)
    
    print("\n[BÁO CÁO KẾT QUẢ]")
    print(f" - Tổng số cửa sổ trích xuất : {len(X)}")
    print(f" - Số lượng nhãn 0 (No-Blink)    : {np.sum(y==0)}")
    print(f" - Số lượng nhãn 1 (Blink)       : {np.sum(y==1)}")
    print(f" - Số lượng nhãn 2 (Long-Closure): {np.sum(y==2)}")
    
    print("\n5. Đang lưu kết quả ra mảng Numpy...")
    os.makedirs(output_dir, exist_ok=True)
    np.save(os.path.join(output_dir, 'X_train_seq.npy'), X)
    np.save(os.path.join(output_dir, 'y_train_seq.npy'), y)
    print(f"Hoàn thành! File được lưu tại: {output_dir}")


In [ ]:
# =========================================
# PHẦN CHẠY KỊCH BẢN TRÊN MÔI TRƯỜNG KAGGLE
# =========================================

# Đổi đường dẫn này nếu folder Dataset trên Kaggle của bạn tên khác nhé
INPUT_CSV = '/kaggle/input/ndhoang2310/eyes-blink-datasetdap-project/dataset_master/metadata.csv'
OUTPUT_DIR = '/kaggle/working/train_data'

process_sequences(
    input_csv=INPUT_CSV, 
    output_dir=OUTPUT_DIR, 
    window_size=7, 
    ear_threshold=0.1050
)